# LLM Fine-Tuning Deep Dive, Part 3 of 3: Comparison & Decision

> **This is the capstone of a three-notebook fine-tuning arc:**
>
> 1. [Part 1: Data-based techniques](01-llm-finetuning-data-techniques.ipynb) changed **what behavior the training objective teaches**: continued pretraining, supervised fine-tuning (SFT), and preference alignment (DPO).
> 2. [Part 2: Parameter-based techniques](02-llm-finetuning-parameter-techniques.ipynb) changed **how much of the model moves while learning**: full fine-tuning, partial freezing, LoRA, and the QLoRA/quantization path.
> 3. **Part 3 (this notebook)** reconnects those axes, inspects the saved candidates, and decides what evidence is still required for each Riverside workload.

## Where We Are: The Fine-Tuning Roadmap

Parts 1 and 2 answered different questions:

- The **data objective** chooses the capability to teach: domain continuation, instruction following, or preference alignment.
- The **parameter strategy** chooses the update scope and artifact shape: all weights, selected layers, or small adapters.
- Part 3 asks which candidate fits a workload and whether the available evidence is strong enough to promote it.

```mermaid
flowchart TD
    Start["Starting point<br/>Base SmolLM2-135M-Instruct: capable, but domain-blind"]

    subgraph Data["Part 1: What should the model learn?"]
        direction TB
        C1["[Done] Concept 1<br/>Continued pretraining<br/>Catalog language and style"]
        C2["[Done] Concept 2<br/>SFT<br/>Instruction following"]
        C3["[Attempted] Concept 3<br/>DPO<br/>Small run was inconclusive"]
        C1 --> C2 --> C3
    end

    subgraph Params["Part 2: How much should move?"]
        direction TB
        C4["[Done] Concept 4<br/>Full fine-tuning<br/>All weights updated"]
        C5["[Done] Concept 5<br/>Partial freezing<br/>Selected layers updated"]
        C6["[Done] Concept 6<br/>LoRA<br/>Small adapter updated"]
        C7["[Explored] Concept 7<br/>QLoRA + quantization<br/>Scaling/deployment path"]
        C4 --> C5 --> C6 --> C7
    end

    Start --> C1
    Start --> C4
    C3 --> Evidence["Part 3: Inspect evidence<br/>without overstating it"]
    C7 --> Evidence
    Evidence --> Retrieval["Knowledge base<br/>Retrieval + grounding"]
    Evidence --> Assistant["Editing assistant<br/>SFT candidate + task gates"]
    Evidence --> Continuation["House-style continuation<br/>Matched retraining required"]
```

> The arrows show the **learning roadmap**, not literal checkpoint ancestry. Continued pretraining and SFT start from separate base models in the demonstrations; DPO continues from the SFT adapter.

| Roadmap stop | What the arc established | Evidence still needed |
| --- | --- | --- |
| Concepts 1-3: data objectives | Each recipe targets a different behavior; the short DPO run is inconclusive | Workload-specific task scores and real preference labels |
| Concepts 4-7: parameter strategies | Each update strategy and artifact pattern is demonstrated | A matched train/test protocol for causal quality and cost comparisons |
| Part 3: decision | The candidates can be inspected under shared diagnostics | Clean test data, repeated task suites, safety, latency, and cost gates |

## How This Capstone Builds the Decision

The notebook follows one evidence ladder:

1. **Inventory** the saved candidates and place each on the data-objective x parameter-strategy map.
2. **Observe examples** under shared prompts, using a concrete reading rubric.
3. **Inspect mechanism** with complete-phrase probability shifts.
4. **Probe corpus fit** on a shared later-chapter sample while disclosing training contamination.
5. **Map missing evidence** with reasoned ablations and workload-specific release gates.
6. **Hand off a shortlist**, not a fabricated universal winner.

Metric mechanics stay brief here. [LLM Evaluation, Part 1](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb) teaches perplexity and the broader automated-metric toolkit; later evaluation notebooks cover judges, safety, hallucination detection, and calibration.

## Table of Contents

1. [Setup: Reloading All Six Trained Checkpoints](#setup-reloading-all-six-trained-checkpoints)
2. [Comparing the Candidates Without Confusing the Axes](#comparing-the-candidates-without-confusing-the-axes)
3. [Decision Time: What Evidence Would Support a Choice?](#decision-time-what-evidence-would-support-a-choice)
4. [Side-by-Side: Every Checkpoint on the Same Prompt](#side-by-side-every-checkpoint-on-the-same-prompt)
5. [Closer Read: One Prompt Without Truncation](#closer-read-one-prompt-without-truncation)
6. [Behavior Evidence: Which Continuations Became Less Surprising?](#behavior-evidence-which-continuations-became-less-surprising)
7. [Shared Corpus Probe](#shared-corpus-probe-diagnostic-perplexity-not-a-holdout)
8. [Technique Combination Grid](#technique-combination-grid-data-x-parameter)
9. [Ablation Study](#ablation-study-what-happens-if-you-skip-a-stage)
10. [What This Fine-Tuning Arc Established](#what-this-fine-tuning-arc-established)
11. [The Decision](#the-decision-what-can-riverside-hand-off-today)

---

## Setup: Reloading All Six Trained Checkpoints

Parts 1 and 2 ran in separate kernels and saved candidates under `./checkpoints/`, so this notebook reloads fresh Python objects rather than relying on hidden state.

> **Prerequisite:** Rerun Parts 1 and 2 from clean kernels with `HuggingFaceTB/SmolLM2-135M-Instruct` before running this notebook. Existing previous-model artifacts are architecture-incompatible with SmolLM2 and cannot be reloaded here; this notebook does not modify or convert checkpoint files.

Two details prevent misleading comparisons:

- `./checkpoints/instruction-lora` was saved before DPO, while `./checkpoints/preference-dpo` was saved afterward.
- The six candidates do not form one controlled sequence. Some differ by objective, parameter strategy, training corpus, and hyperparameters. The comparison labels those limits explicitly.


## Choose the Training Path

![Fine-tuning decision matrix matching adaptation needs to continued pretraining, SFT plus LoRA, DPO plus LoRA, or full fine-tuning](images/finetuning-decision-matrix.png)

Use the matrix as a first-pass prompt, not a set of exclusive branches:

1. Choose the **objective** from the behavior the workload needs.
2. Choose the **parameter strategy** from matched quality, training resources, and artifact constraints.
3. Measure serving latency separately; update strategy alone does not determine inference speed.

No candidate is promoted from this diagram or from the shared corpus probe. Promotion requires clean task, safety, latency, and cost evidence.

> **PyTorch → Keras:** `torch.cuda.is_available()` + `.to(device)` explicitly move a model/tensors to
> GPU or CPU, `AutoModelForCausalLM.from_pretrained(...)` loads pretrained weights, and
> `model.generate(...)` run inside `torch.no_grad()` performs autoregressive decoding without tracking
> gradients (nothing to backprop through during inference). **Keras/TF equivalent:** TensorFlow places
> ops on GPU automatically (explicit placement is `tf.device(...)`, rarely needed); the loading call
> would be `TFAutoModelForCausalLM.from_pretrained(...)` followed by the same `.generate(...)` method --
> Keras/TF has no separate "no_grad" context since inference doesn't build a gradient tape by default.


In [ ]:
# Re-establish Parts 1-2's foundations using the same SmolLM2 base and prompt contract.
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
SYSTEM_PROMPT = "You are a careful fiction-writing assistant for Riverside Publishing."
CONTINUATION_INSTRUCTION = "Continue the fiction narrative in the same style."

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"

num_hidden_layers = base_model.config.num_hidden_layers
hidden_size = base_model.config.hidden_size
total_base_parameters = sum(parameter.numel() for parameter in base_model.parameters())
print(
    f"Loaded {MODEL_NAME}: {total_base_parameters:,} parameters, "
    f"{num_hidden_layers} decoder layers, hidden size {hidden_size}."
)


def instruction_prompt(prompt):
    """Build the user message used by the SFT and DPO recipes in Parts 1-2."""
    return f"{CONTINUATION_INSTRUCTION}\n\n{prompt}"


def apply_instruction_template(prompt):
    """Serialize an instruction through the model's native chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def generate(model, prompt, max_new_tokens=60, use_chat_template=False):
    """Generate only new tokens, using the native chat format for instruction candidates."""
    model.eval()
    model_input = apply_instruction_template(prompt) if use_chat_template else prompt
    inputs = tokenizer(model_input, return_tensors="pt").to(device)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        output_ids[0][prompt_length:], skip_special_tokens=True
    ).strip()
    return completion if completion else "[model stopped immediately after the prompt]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")


### Reloading the Five Fine-Tuned Checkpoints

Each PEFT-wrapped adapter (instruction-tuned LoRA, DPO, LoRA continued pretraining) gets its own fresh
base-model instance rather than sharing `base_model` above -- the same "every PEFT wrapper gets its
own base" rule Parts 1-2 followed throughout. `freeze_model`'s `requires_grad` flags are re-applied
after loading (see the comment below) since that bookkeeping isn't part of a saved checkpoint -- only
the trained weights are.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, path)` wraps a fresh base model with a
> saved LoRA adapter's weights; `named_parameters()` iterates `(name, tensor)` pairs so `requires_grad`
> can be toggled per-parameter (used here to re-apply the freeze pattern, since that bookkeeping isn't
> part of a saved checkpoint), and `p.numel()` counts a tensor's elements to total trainable params.
> **Keras/TF equivalent:** LoRA loading has no single standard TF API (usually a custom `tf.keras.Model`
> subclass or a TF-specific PEFT integration); freezing is coarser-grained -- `layer.trainable = False`
> per layer rather than per-parameter -- and element counts come from `tf.size(variable)`.


In [ ]:
# Reload only artifacts regenerated by Parts 1-2 for MODEL_NAME.
non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to(device)

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, "./checkpoints/instruction-lora"
).to(device)

dpo_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
policy_model = PeftModel.from_pretrained(
    dpo_base_reload, "./checkpoints/preference-dpo"
).to(device)

freeze_model = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/partial-freeze"
).to(device)
n_layers = freeze_model.config.num_hidden_layers
unfreeze_from = n_layers - max(2, n_layers // 4)

for parameter in freeze_model.parameters():
    parameter.requires_grad = False
for layer in freeze_model.model.layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.model.norm.parameters():
    parameter.requires_grad = True
for parameter in freeze_model.lm_head.parameters():
    parameter.requires_grad = True

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = PeftModel.from_pretrained(
    lora_pt_base, "./checkpoints/peft-lora"
).to(device)

adapter_models = {
    "instruction LoRA": instruct_lora_model,
    "DPO policy": policy_model,
    "continued-pretraining LoRA": lora_pt_model,
}
expected_targets = set(LORA_TARGET_MODULES)
for adapter_name, adapter_model in adapter_models.items():
    configured_targets = {
        target
        for peft_config in adapter_model.peft_config.values()
        for target in peft_config.target_modules
    }
    if configured_targets != expected_targets:
        raise ValueError(
            f"{adapter_name} targets {sorted(configured_targets)}, expected "
            f"{sorted(expected_targets)}. Rerun Parts 1-2 with the SmolLM2 LoRA recipe."
        )

for model in (
    non_instruct_ckpt,
    instruct_lora_model,
    policy_model,
    freeze_model,
    lora_pt_model,
):
    model.eval()

# Derive every parameter budget from the models actually loaded above.
total_params = sum(parameter.numel() for parameter in base_model.parameters())
full_ft_params = sum(parameter.numel() for parameter in non_instruct_ckpt.parameters())
partial_ft_params = sum(
    parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad
)
lora_params = sum(
    parameter.numel()
    for name, parameter in instruct_lora_model.named_parameters()
    if ".lora_" in name
)
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]

trainable_partial_names = [
    name for name, parameter in freeze_model.named_parameters() if parameter.requires_grad
]
assert any(
    name.startswith(f"model.layers.{unfreeze_from}.")
    for name in trainable_partial_names
), "Expected SmolLM2 trailing-layer parameter names were not found"

print("Reloaded all six candidates (baseline + 5 fine-tuned):")
print(f"  SmolLM2 layers/hidden:    {n_layers} / {freeze_model.config.hidden_size}")
print(f"  Full fine-tuning:      {full_ft_params:,} parameters ({param_pcts[0]:.2f}%)")
print(f"  Partial freezing:      {partial_ft_params:,} parameters ({param_pcts[1]:.2f}%)")
print(f"  LoRA matrices:         {lora_params:,} parameters ({param_pcts[2]:.2f}%)")

In [ ]:
# Corpus loader (needed for the held-out perplexity + ablation sections further down) and
# visualization imports used throughout this notebook.
# Resolve the notebook's own directory, falling back to cwd if the VS Code variable is unavailable
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"

# If the sibling content folder isn't found here, try resolving it from the repo root instead
if not CONTENT_DIR.exists():
    _fallback = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if _fallback.exists():
        CONTENT_DIR = _fallback

print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel corpus filenames keyed by genre alias, shared by every corpus-loading helper below
NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}


# Load paragraphs long enough to be useful, from each requested novel's earliest chapters
def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from the multi-novel corpus (identical to Parts 1-2)."""
    if novels is None:
        novels = list(NOVELS.keys())
    paragraphs = []

    # Walk every requested novel and pull paragraphs from its earliest chapters
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")

            # Split on blank lines and keep only paragraphs above the minimum length
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

# Silence noisy library warnings so they don't clutter notebook output
warnings.filterwarnings("ignore")

# Keep figure resolution/font size consistent across every plot in this notebook
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Corpus loader and visualization imports ready.")


---



## Comparing the Candidates Without Confusing the Axes



A label such as "SFT + LoRA" contains **two choices**:



- **SFT** says what loss and data format taught the model.

- **LoRA** says which parameters were allowed to change.



There is a third ingredient too: the exact corpus slice and training recipe. A fair causal comparison must hold those fixed as well.



### Comparison A: Same objective family, but not a controlled ablation



The three candidates below all used next-token prediction on Riverside prose. That makes them useful examples of the parameter strategies, but their upstream runs used different novel subsets, chapter counts, learning rates, and step budgets.



| Continued-pretraining candidate | Nominal parameters updated | What we may inspect here | What we may not conclude |

| --- | ---: | --- | --- |

| Full fine-tuning (Concepts 1 and 4) | 100% | Its behavior, corpus-probe score, and artifact cost | That full FT caused any score gap |

| Partial freezing (Concept 5) | derived at runtime from the SmolLM2 layer-freezing rule | The same descriptive signals | That partial freezing is second-best |

| LoRA (Concept 6) | derived at runtime from the loaded SmolLM2 adapter | The same descriptive signals | The quality cost of LoRA in isolation |



A proper parameter ablation would train all three on the **same ordered examples**, with matched token/step budgets, seed policy, evaluation split, and a documented learning-rate protocol. Part 3 therefore calls the current numbers a **candidate comparison**, not an isolated parameter-strategy result.



### Comparison B: Different objectives answer different workload questions



The data-axis checkpoints are intentionally not interchangeable:



| Candidate | Objective | Parameter strategy | Behavior to look for |

| --- | --- | --- | --- |

| Continued pretraining | Next-token prediction on raw prose | Full fine-tuning | Domain language and house-style continuation |

| Instruction tuning | Supervised prompt/completion loss | LoRA | Direct instruction following |

| Preference alignment | DPO on chosen/rejected pairs | LoRA on the SFT adapter | Movement toward editor preferences |



This is a practical capability progression, not a leaderboard. A continuation model can be better at predicting prose while an SFT model is better at answering an instruction. The short DPO run is retained as an inconclusive candidate, not presented as successful alignment.



### Comparison C: Cells we did not train



The full design space is a 3 x 3 grid: three objectives by three parameter strategies. This run populated five cells. SFT + full fine-tuning, SFT + partial freezing, DPO + full fine-tuning, and DPO + partial freezing remain unmeasured; later visuals keep them blank instead of inventing results.



QLoRA is also a valid parameter path, but Part 2 explored its mechanics rather than training a seventh candidate. It informs a future scaling plan without entering the measured checkpoint comparison.



## Decision Time: What Evidence Would Support a Choice?



Before generating anything, define what each signal can answer:



| Evidence | Question it can answer | Question it cannot answer alone |

| --- | --- | --- |

| Same-prompt generations | What behavior appears in these examples? | Whether it generalizes across prompts or samples |

| Corpus-term examples | Does output contain expected catalog language? | Whether claims are consistently correct and grounded |

| Phrase probability shift | Did training move probability toward selected continuations? | Whether a complete response is useful or safe |

| Shared corpus probe | How surprised is each candidate by the same later-chapter sample? | A clean generalization ranking when training overlap differs |

| Trainable-parameter count | How much state was updated during that recipe? | Peak memory, wall-clock time, serving latency, or quality |

| Reasoned ablation map | What failure should a matched experiment test? | The size of the effect without retraining |



The last column is the notebook's guardrail. These demonstrations build intuition and identify candidates; they do not replace the clean workload-specific evaluation suite introduced in the evaluation chapters.



### Side-by-Side: Every Checkpoint on the Same Prompt



Start with the cheapest evidence: place every candidate under the same catalog prompt. Read outputs diagnostically rather than asking which paragraph merely sounds nicest:



1. Does it use Riverside-specific language rather than generic genre prose?

2. Does it obey the requested task and stop at an appropriate point?

3. Does the DPO candidate differ consistently from SFT, or does the small preference run remain inconclusive?



Because generation can be stochastic, one output is an example, not a score. The later sections move from examples to progressively more structured diagnostics.

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained("./checkpoints/...")` reloads a saved
> fine-tuned checkpoint from disk into a fresh `torch.nn.Module`, then `.to(device)` places it on
> GPU/CPU before the loop below calls the `generate()` helper defined earlier on each model in turn.
> **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(path)` loads the same checkpoint
> format into a `tf.keras.Model`; TensorFlow doesn't need an explicit `.to(device)` call since device
> placement is handled by default device scoping (or `tf.distribute` for multi-device setups) instead.


In [ ]:
# Compare every candidate on shared catalog prompts.
COMPARISON_PROMPTS = {
    "scifi": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy": "Kerra Valmont felt all five tides simultaneously as",
    "mystery": "Elena Voss studied the 1879 survey map and realized",
    "cyberpunk": "In the Lower Stacks of Neo-Shanghai, Kai Chen",
}

models_to_test = {
    "Baseline (no fine-tuning)": base_model,
    "Continued pretraining (full FT)": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial fine-tuning": freeze_model,
    "PEFT LoRA continued pretraining": lora_pt_model,
}

print("=" * 80)
print("QUALITATIVE CANDIDATE EXAMPLES - SAME PROMPTS, ONE SAMPLE EACH")
print("=" * 80)

for prompt_name, prompt in COMPARISON_PROMPTS.items():
    print(f'\nPrompt ({prompt_name}): "{prompt}"')
    print("-" * 80)

    for model_index, (model_name, model) in enumerate(models_to_test.items()):
        sample_seed = 42 + model_index
        torch.manual_seed(sample_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(sample_seed)

        uses_chat_template = "Instruction" in model_name or "Preference" in model_name
        effective_prompt = instruction_prompt(prompt) if uses_chat_template else prompt
        output = generate(
            model,
            effective_prompt,
            max_new_tokens=60,
            use_chat_template=uses_chat_template,
        )
        output_display = output[:120] + "..." if len(output) > 120 else output

        format_note = " + SmolLM2 chat template" if uses_chat_template else ""
        print(f"\n[{model_name}] seed={sample_seed}{format_note}")
        print(f"  Output: {output_display}")

print("\n" + "=" * 80)
print("READ THESE AS EXAMPLES:")
print("1. Catalog language: are names and setting details specific rather than generic?")
print("2. Task behavior: does the model continue prose or answer an instruction?")
print("3. Stability: a claim requires repeated prompts/samples and a scoring rubric.")
print("=" * 80)

## Closer Read: One Prompt Without Truncation

The prompt matrix above improves breadth but truncates each sample for scanning. Now inspect one complete example from every candidate.

Use the same three-question rubric:

| Lens | Concrete sign to look for | Common false positive |
| --- | --- | --- |
| Catalog language | Story-specific entities or relationships used coherently | Repeating a name copied from the prompt |
| Task behavior | Direct answer or bounded continuation in the requested format | Fluent prose that ignores the instruction |
| Preference signal | A consistent difference between SFT and DPO across examples | One nicer-sounding sample caused by decoding randomness |

The cell below is still qualitative. It exposes the full inputs, including the instruction prefix, so differences caused by prompt format are not mistaken for differences caused by weights.

In [ ]:
# Instruction and preference candidates use the chat contract from Parts 1-2.
instruct_prompt = instruction_prompt(PROMPT)
print(f"Shared prompt (plain models)       : {PROMPT!r}")
print(f"Shared user request (chat models) : {instruct_prompt!r}")
print()

print("=== Baseline (no fine-tuning) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(base_model, PROMPT)}")
print()

print("=== Non-instructional continued pretraining (full fine-tune) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(non_instruct_ckpt, PROMPT)}")
print()

print("=== Instruction-tuned (LoRA) ===")
print(f"  Input : SmolLM2 chat template over {instruct_prompt!r}")
print(
    f"  Output: {generate(instruct_lora_model, instruct_prompt, use_chat_template=True)}"
)
print()

print("=== Preference-aligned (DPO on the instruction-tuned adapter) ===")
print(f"  Input : SmolLM2 chat template over {instruct_prompt!r}")
print(f"  Output: {generate(policy_model, instruct_prompt, use_chat_template=True)}")
print()

print("=== Partial fine-tuning (layer freezing) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(freeze_model, PROMPT)}")
print()

print("=== Parameter-efficient (LoRA continued pretraining) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(lora_pt_model, PROMPT)}")
print()

## Behavior Evidence: Which Continuations Became Less Surprising?

A generated paragraph can look strong or weak because of one decoding sample. Phrase probabilities let us inspect a smaller mechanism directly.

For a fixed prompt, score several complete candidate continuations under the baseline and continued-pretraining checkpoint. The score is the **mean log-probability per continuation token**:

$$
\text{mean log-probability} = \frac{1}{T}\sum_{t=1}^{T}\log p(w_t \mid \text{prompt}, w_{<t})
$$

A higher value means the phrase is less surprising to the model. Averaging per token makes phrases of different token lengths more comparable.

### Concrete prediction

After `Aria Voss checked the Meridian's Promise and ...`, continued pretraining should make catalog-flavored phrases such as "the Keeper's maintenance logs" or "the quantum fold drive" less surprising. Generic grammatical continuations may move less.

This remains a **local diagnostic**. It can show that training shifted probability toward selected phrases; it cannot establish factual correctness, instruction following, safety, or overall response quality. [Calibration and Confidence Evaluation](../05-llm-evaluation/04-calibration-and-confidence.ipynb) later explains why model probability is not calibrated correctness.

> **PyTorch → Keras:** `model.eval()` switches dropout/batch-norm-style layers to inference behavior;
> `torch.no_grad()` disables gradient tracking for the forward pass below; calling `model(**inputs)`
> runs a forward pass and returns `outputs.logits` (raw scores), and `F.softmax(logits, dim=-1)`
> (from `torch.nn.functional`) converts those logits into a probability distribution over the vocabulary.
> **Keras/TF equivalent:** Keras layers infer train/inference behavior automatically (or via a
> `training=False` argument) instead of an explicit `.eval()` call, and there's no separate "no_grad"
> context since plain forward calls outside a `GradientTape` don't track gradients; the softmax step is
> `tf.nn.softmax(logits, axis=-1)` -- same idea, `axis` instead of `dim`.


In [ ]:
import torch.nn.functional as F

prompt_for_analysis = "Aria Voss checked the Meridian's Promise and"

# Full phrases, not first-token proxies. Domain phrases use Riverside-specific concepts;
# controls are plausible but generic continuations.
candidate_phrases = {
    "Keeper's maintenance logs": " opened the Keeper's maintenance logs",
    "quantum fold drive": " checked the quantum fold drive",
    "containment-field anomaly": " detected a containment-field anomaly",
    "looked at the screen": " looked at the screen",
    "said nothing": " said nothing",
    "went back to work": " went back to work",
}
domain_labels = set(list(candidate_phrases)[:3])


def continuation_logprob(model, prompt, continuation):
    """Return mean/sum log-probability for every token in one continuation."""
    prompt_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = tokenizer(
        prompt + continuation, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    prompt_length = prompt_ids.shape[1]

    model.eval()
    with torch.no_grad():
        logits = model(full_ids).logits[:, :-1, :]
        log_probs = F.log_softmax(logits, dim=-1)

    # Position prompt_length-1 predicts the first continuation token.
    continuation_ids = full_ids[:, prompt_length:]
    continuation_log_probs = log_probs[
        0, prompt_length - 1 : full_ids.shape[1] - 1
    ].gather(1, continuation_ids[0].unsqueeze(1)).squeeze(1)

    return {
        "mean": continuation_log_probs.mean().item(),
        "sum": continuation_log_probs.sum().item(),
        "tokens": continuation_ids.shape[1],
    }


phrase_results = []
for label, phrase in candidate_phrases.items():
    baseline_score = continuation_logprob(base_model, prompt_for_analysis, phrase)
    finetuned_score = continuation_logprob(
        non_instruct_ckpt, prompt_for_analysis, phrase
    )
    phrase_results.append(
        {
            "label": label,
            "kind": "catalog" if label in domain_labels else "generic control",
            "tokens": finetuned_score["tokens"],
            "baseline": baseline_score["mean"],
            "finetuned": finetuned_score["mean"],
            "delta": finetuned_score["mean"] - baseline_score["mean"],
        }
    )

print(f"Prompt: {prompt_for_analysis!r}\n")
print(f"{'Phrase':30} {'Type':16} {'Tok':>3} {'Base':>9} {'Fine-tuned':>11} {'Delta':>9}")
print("-" * 86)
for row in phrase_results:
    print(
        f"{row['label']:30} {row['kind']:16} {row['tokens']:>3} "
        f"{row['baseline']:>9.3f} {row['finetuned']:>11.3f} {row['delta']:>+9.3f}"
    )

labels = [row["label"] for row in phrase_results]
baseline_values = [row["baseline"] for row in phrase_results]
finetuned_values = [row["finetuned"] for row in phrase_results]
deltas = [row["delta"] for row in phrase_results]
positions = np.arange(len(labels))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.35, 1]})

axes[0].barh(
    positions - width / 2,
    baseline_values,
    height=width,
    label="Baseline",
    color="#4C78A8",
)
axes[0].barh(
    positions + width / 2,
    finetuned_values,
    height=width,
    label="Continued pretraining",
    color="#E45756",
)
axes[0].set_yticks(positions)
axes[0].set_yticklabels(labels)
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean log-probability per token (higher = less surprising)")
axes[0].set_title("Same Prompt, Complete Phrase Scores")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.25)

delta_colors = ["#2A9D8F" if value >= 0 else "#D1495B" for value in deltas]
axes[1].barh(positions, deltas, color=delta_colors)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(labels)
axes[1].invert_yaxis()
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Change after continued pretraining (nats/token)")
axes[1].set_title("Probability Shift, Without a Quality Claim")
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

catalog_deltas = [row["delta"] for row in phrase_results if row["kind"] == "catalog"]
control_deltas = [
    row["delta"] for row in phrase_results if row["kind"] == "generic control"
]
print(
    f"\nMean catalog-phrase shift: {np.mean(catalog_deltas):+.3f} nats/token\n"
    f"Mean generic-control shift: {np.mean(control_deltas):+.3f} nats/token"
)
print(
    "Interpretation: positive shifts mean these selected phrases became less surprising. "
    "They do not prove that generated claims are correct."
)

## Shared Corpus Probe: Diagnostic Perplexity, Not a Holdout

The phrase analysis examined a handful of selected continuations. We now broaden the view to later-chapter prose and ask how surprised each candidate is by the same text.

Perplexity is the exponential of mean next-token negative log-likelihood. Lower means better fit to the evaluated prose. [LLM Evaluation, Part 1](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb#part-5--perplexity-how-surprised-is-the-model-by-its-own-output) derives the metric in depth.

### Evidence boundary

This is **not a clean held-out evaluation**. The upstream demonstrations did not share one file-level train/evaluation split:

- Full fine-tuning used all available chapters from selected novels, including some chapters sampled here.
- Partial freezing and LoRA used smaller, different novel/chapter subsets.
- SFT and DPO used different data formats and coverage again.

The later-chapter sample is therefore a **shared corpus probe**. It is useful for seeing what each saved model assigns probability to, but its ranking mixes training exposure, objective, parameter strategy, and hyperparameters. It must not select a production winner.

### What the code improves

- Every candidate receives exactly the same probe paragraphs.
- Negative log-likelihood is aggregated by evaluated token, so a short paragraph does not count as much as a long one.
- The chart labels the result as descriptive and keeps the contamination warning visible.

A clean comparison requires defining the file-level split before training and retraining every candidate only on the training partition. Until then, use this plot to build metric intuition, not to claim generalization.

> **PyTorch → Keras:** passing `labels=enc["input_ids"]` into the model's forward call makes the
> Hugging Face model compute cross-entropy loss internally and return it as `out.loss` (a scalar
> tensor); `torch.no_grad()` skips gradient tracking since this is evaluation-only, and `.item()`
> pulls the plain Python float out of that 0-d tensor, which `math.exp(loss)` then turns into
> perplexity. **Keras/TF equivalent:** the TF counterpart model supports the same `labels=` convenience
> (`model(enc, labels=...)`), while plain Keras code would instead call
> `tf.keras.losses.SparseCategoricalCrossentropy()(y_true, logits)` and use `.numpy()` in place of
> `.item()` to extract the scalar.


In [ ]:
import math


# Use the same later-chapter sample for every candidate. This is a shared probe, not a clean holdout,
# because upstream training coverage differs and full FT saw some of these files.
def load_corpus_probe(start_chapter_index=10, chapters_per_novel=2, min_len=200):
    paragraphs = []
    source_files = []

    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        probe_files = chapter_files[
            start_chapter_index : start_chapter_index + chapters_per_novel
        ]
        source_files.extend(probe_files)

        for path in probe_files:
            text = path.read_text(encoding="utf-8")
            for paragraph in text.split("\n\n"):
                paragraph = paragraph.strip().replace("\n", " ")
                if len(paragraph) >= min_len:
                    paragraphs.append(paragraph)

    return paragraphs, source_files


probe_paragraphs, probe_files = load_corpus_probe()
print(
    f"Shared corpus probe: {len(probe_paragraphs)} paragraphs from "
    f"{len(probe_files)} later-chapter files."
)
print(
    "WARNING: this is descriptive, not held out. Upstream candidates used different "
    "training files, and full fine-tuning saw some probe chapters."
)


# Aggregate negative log-likelihood by evaluated token rather than averaging paragraph means.
def compute_corpus_probe(model, paragraphs, max_length=128):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for paragraph in paragraphs:
            encoded = tokenizer(
                paragraph,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            output = model(**encoded, labels=encoded["input_ids"])

            # Causal loss predicts tokens 2..N from tokens 1..N-1.
            valid_tokens = int(encoded["attention_mask"][:, 1:].sum().item())
            total_nll += output.loss.item() * valid_tokens
            total_tokens += valid_tokens

    mean_nll = total_nll / total_tokens
    return {
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "tokens": total_tokens,
    }


models_for_probe = {
    "Baseline (no fine-tuning)": base_model,
    "Full fine-tuning": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial freezing": freeze_model,
    "LoRA continued pretraining": lora_pt_model,
}

print(f"\n{'=' * 72}\nShared corpus-probe perplexity (descriptive only):\n{'=' * 72}")
corpus_probe_results = {}
for name, model in models_for_probe.items():
    result = compute_corpus_probe(model, probe_paragraphs)
    corpus_probe_results[name] = result
    print(
        f"  {name:<32} NLL={result['mean_nll']:6.3f}  "
        f"PPL={result['perplexity']:8.1f}  tokens={result['tokens']:,}"
    )
print(f"{'=' * 72}")

probe_ranking = sorted(
    corpus_probe_results.items(), key=lambda item: item[1]["perplexity"]
)
names = [name for name, _ in probe_ranking]
perplexities = [result["perplexity"] for _, result in probe_ranking]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names[::-1], perplexities[::-1], color="#4C78A8")
ax.set_xlabel("Corpus-probe perplexity (lower = better fit to this sample)")
ax.set_title(
    "Shared Later-Chapter Corpus Probe\n"
    "Descriptive only: training exposure differs across candidates",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

lowest_name, lowest_result = probe_ranking[0]
print(
    f"Lowest value on this contaminated probe: {lowest_name!r} "
    f"(PPL={lowest_result['perplexity']:.1f})."
)
print(
    "Do not promote that candidate from this ranking. A valid selection requires a split "
    "created before matched training plus workload-specific task metrics."
)

## Technique Combination Grid: Data x Parameter

The two independent choices can now be placed on one map:

- Pick a **row** for the behavior the model should learn.
- Pick a **column** for which parameters the recipe updates.

| Data objective | Full fine-tuning | Partial freeze | LoRA |
| --- | --- | --- | --- |
| Continued pretraining | Trained: `non_instruct_ckpt` | Trained: `freeze_model` | Trained: `lora_pt_model` |
| Instruction tuning (SFT) | Not trained | Not trained | Trained: `instruct_lora_model` |
| Preference alignment (DPO) | Not trained | Not trained | Trained: `policy_model` |

This grid is primarily a **coverage visual**. Grey cells mean "not trained," not "failed."

The left heatmap places shared corpus-probe perplexity into the five populated cells. Those values are descriptive because upstream data exposure and training recipes differ. The right heatmap shows nominal trainable-parameter percentages for each recipe family; it does not measure peak memory, elapsed time, serving latency, or quality.

A future controlled parameter study would populate one row using the same train split, ordered examples, token budget, seed protocol, and evaluation split for full FT, partial freezing, and LoRA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Rows are learning objectives; columns are parameter strategies.
data_objectives = [
    "Continued\nPretraining",
    "Instruction\nTuning (SFT)",
    "Preference\nAlignment (DPO)",
]
parameter_strategies = [
    "Full FT\n(100%)",
    "Partial Freeze\n(~21%)",
    "LoRA\n(<1%)",
]

# Only five of the nine possible recipe cells were trained.
checkpoint_map = {
    (0, 0): "Full fine-tuning",
    (0, 1): "Partial freezing",
    (0, 2): "LoRA continued pretraining",
    (1, 2): "Instruction-tuned (LoRA)",
    (2, 2): "Preference-aligned (DPO)",
}
trained_mask = np.zeros((3, 3), dtype=bool)
probe_grid = np.full((3, 3), np.nan)
for (row, column), checkpoint_name in checkpoint_map.items():
    trained_mask[row, column] = True
    if checkpoint_name in corpus_probe_results:
        probe_grid[row, column] = corpus_probe_results[checkpoint_name]["perplexity"]

# Nominal training-time parameter budgets reconstructed earlier.
parameter_row = [param_pcts[0], param_pcts[1], param_pcts[2]]
parameter_grid = np.array([parameter_row] * 3)


def draw_recipe_grid(axis, values, value_format, title, color_map, legend_label):
    """Draw measured cells and leave untrained combinations visibly blank."""
    display_values = np.where(trained_mask, values, np.nan)
    color_map = plt.get_cmap(color_map).copy()
    color_map.set_bad(color="#D9D9D9")

    valid_values = display_values[trained_mask]
    value_min = valid_values.min()
    value_max = valid_values.max()
    if value_min == value_max:
        value_max = value_min + 1

    image = axis.imshow(
        display_values,
        cmap=color_map,
        vmin=value_min,
        vmax=value_max,
        aspect="auto",
    )

    for row in range(3):
        for column in range(3):
            if trained_mask[row, column]:
                axis.text(
                    column,
                    row,
                    value_format.format(display_values[row, column]),
                    ha="center",
                    va="center",
                    fontsize=11,
                    fontweight="bold",
                )
            else:
                axis.text(
                    column,
                    row,
                    "not\ntrained",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="#666666",
                    style="italic",
                )

    axis.set_xticks(range(3))
    axis.set_yticks(range(3))
    axis.set_xticklabels(parameter_strategies, fontsize=9)
    axis.set_yticklabels(data_objectives, fontsize=9)
    axis.set_xlabel("Parameter strategy", fontweight="bold")
    axis.set_ylabel("Data objective", fontweight="bold")
    axis.set_title(title, fontsize=11, fontweight="bold", pad=8)
    plt.colorbar(image, ax=axis, fraction=0.04, pad=0.04, label=legend_label)


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Data Objective x Parameter Strategy: Coverage and Descriptive Signals",
    fontsize=13,
    fontweight="bold",
)

draw_recipe_grid(
    axes[0],
    probe_grid,
    "{:.1f}",
    "Shared Corpus-Probe Perplexity\n(confounded; not a model ranking)",
    "YlOrRd_r",
    "Probe perplexity",
)
draw_recipe_grid(
    axes[1],
    parameter_grid,
    "{:.2f}%",
    "Nominal Parameters Updated\n(not peak memory or latency)",
    "Blues_r",
    "Updated parameters (%)",
)

plt.tight_layout()
plt.show()

print("How to read the two panels:")
print("1. Topology: five recipe cells exist; four combinations were not trained.")
print("2. Left: probe values mix data exposure, objective, hyperparameters, and strategy.")
print("3. Right: parameter percentages describe update scope, not end-to-end resource cost.")
print("4. No quality/efficiency frontier can be inferred until the continued-pretraining row is retrained under a matched protocol.")

## Ablation Study: What Happens If You Skip a Stage?

An ablation removes one component while holding the rest of the experiment fixed. That standard is important here.

Only Experiment 5 below is executed in this notebook, and it uses one prompt. Experiments 1-4 are **reasoned counterfactuals**: useful hypotheses, not measured ablation results. A production study would retrain each variant with matched data, seeds, token budgets, and evaluation suites.

### Failure map

| Change | Capability at risk | Expected symptom | Evidence status |
| --- | --- | --- | --- |
| Skip continued pretraining | Familiarity with private catalog language | Generic continuations and weak domain terminology | Hypothesis; not retrained here |
| Skip SFT | Instruction-following format | Continues the prompt instead of answering directly | Hypothesis; not retrained here |
| Skip DPO | Learned editor preference | SFT behavior remains, but preferred distinctions may weaken | Hypothesis; current DPO run did not establish benefit |
| Run DPO before later SFT | Retention of the preference update | Later SFT may move away from the preference optimum | Hypothesis; regression is possible, not guaranteed |
| Use full fine-tuning everywhere | Parameter efficiency and artifact flexibility | Higher optimizer state, storage, and rollback burden | Engineering hypothesis; exact cost depends on runtime |
| Skip fine-tuning | Learned private-domain behavior | Prompting cannot recall facts absent from weights or context | Illustrated with one prompt below |

### 1. Skip continued pretraining

**Question:** Can Riverside teach instruction following without first adapting to raw manuscript prose?

Yes. SFT can teach the response format and facts represented in its demonstrations. What it may lack is broad corpus coverage and house-style continuation if those examples are absent from the SFT set. Continued pretraining may also be unnecessary when the base already knows the domain language or retrieval supplies private evidence at request time.

**To measure it:** train matched SFT adapters from the base and from a continued-pretrained base, then compare continuation, instruction following, and grounded QA separately.

### 2. Skip instruction tuning

**Question:** Can preference optimization turn a prose continuation model directly into an assistant?

DPO learns relative preferences among candidate responses; it does not automatically provide broad instruction-following coverage. Without SFT, the candidates may not yet have the direct-answer and stopping behavior Riverside wants to rank. That is a likely capability mismatch, not a mathematical rule that DPO always requires SFT.

**To measure it:** compare DPO initialized from a base/continued-pretrained model with DPO initialized from the SFT adapter, using the same preference pairs and task suite.

### 3. Run DPO before a later SFT pass

**Question:** Will later supervised training preserve an earlier preference update?

Not necessarily. A later stage can move the same adapter parameters away from earlier behavior, but calling that behavior "erased" would be too strong without measurement. The effect depends on data, learning rates, token budgets, and parameter overlap.

**To measure it:** train both orders under matched budgets and evaluate instruction following plus pairwise editor preference after each checkpoint.

### 4. Use full fine-tuning for every stage

**Question:** Would updating every weight remove the need for LoRA?

It removes the adapter constraint, but it does not guarantee better production quality. Full fine-tuning updates more state and creates larger workload-specific artifacts; LoRA keeps one base reusable and adapters easy to version or swap.

The current arc cannot quantify that quality/cost frontier because the upstream full-FT, partial-freeze, and LoRA runs used different data and hyperparameters.

**To measure it:** retrain on one immutable split and record task quality, peak memory, wall-clock time, artifact size, serving latency, and rollback time.

### 5. Skip fine-tuning and use prompting alone

**Question:** Can a sufficiently careful prompt recover Riverside's unpublished facts?

A prompt can reorganize behavior already available to the model and can provide facts directly in context. It cannot recall a private fact that is present in neither the weights nor the prompt. Riverside therefore has three complementary tools:

- Fine-tuning for durable behavior or style adaptation.
- Retrieval for current, citable private facts.
- Prompting for task instructions and formatting.

The cell below compares an untouched base model with the instruction-tuned candidate on one catalog question. Treat it as an illustration that motivates a benchmark, not as the benchmark itself.

### Putting Experiment 5 to the Test

Let's actually run the zero-shot prompt from Experiment 5 above instead of just reading the
hypothetical example, comparing the untouched base model against the instruction-tuned checkpoint we
trained earlier in this notebook.


In [ ]:
# Experiment 5 in practice: raw prompting versus the regenerated SFT adapter.
zero_shot_prompt = "Who is Aria Voss?"

print("=== Base model, raw prompt (no fine-tuning) ===")
print(generate(base_model, zero_shot_prompt), "\n")

print("=== Instruction-tuned model (SmolLM2 chat format used during SFT) ===")
print(
    generate(
        instruct_lora_model,
        zero_shot_prompt,
        use_chat_template=True,
    )
)

## What This Fine-Tuning Arc Established

The arc implemented real training mechanics, but implementation and evidence are different things. This ledger keeps them separate.

### Built and demonstrated

| Area | What was actually done | What the demonstration supports |
| --- | --- | --- |
| Continued pretraining | Full-FT, partial-freeze, and LoRA recipes saved artifacts | These strategies can adapt a causal LM to manuscript prose |
| Instruction tuning | An SFT LoRA adapter trained on prompt/completion pairs | The recipe targets instruction-formatted behavior |
| Preference alignment | DPO continued from the SFT adapter | The mechanics run; the small preference experiment was inconclusive |
| QLoRA and quantization | QLoRA mechanics explained; CPU dynamic quantization executed | A scaling/deployment path, not a trained QLoRA quality result |
| Candidate examples | Shared prompts generated from six model objects | Concrete behavioral examples worth testing systematically |
| Phrase diagnostic | Complete continuation phrases scored token by token | Selected phrases became more or less surprising after adaptation |
| Corpus probe | All candidates scored on the same later-chapter sample | Descriptive fit to that sample, with known training contamination |
| Recipe grid | Five trained combinations placed on a 3 x 3 map | Which combinations exist and which remain unmeasured |

### Not established by this run

- A causal quality ranking among full fine-tuning, partial freezing, and LoRA.
- Clean held-out perplexity, because the training runs did not share a predeclared file split.
- Reliable instruction-following pass rates across a task suite.
- A DPO preference win-rate improvement over SFT.
- Factual grounding, safety, calibration, serving latency, or end-to-end cost.

Those are not footnotes. They are the exact measurements needed before promotion.

## Further Reading & Scaling Up

A stronger follow-up experiment would:

1. Create immutable train, validation, and test file lists **before** training.
2. Feed full FT, partial freezing, and LoRA the same ordered continued-pretraining examples and token budget.
3. Run multiple seeds and report uncertainty rather than one ordering.
4. Evaluate each workload with its own rubric: continuation fit, instruction pass rate, preference win rate, grounded QA, safety, latency, and cost.
5. Generate the release scorecard directly from those results.

Key references:

- LoRA: [Hu et al. 2021](https://arxiv.org/abs/2106.09685)
- InstructGPT / RLHF: [Ouyang et al. 2022](https://arxiv.org/abs/2203.02155)
- DPO: [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290)
- Instruction tuning / FLAN: [Wei et al. 2021](https://arxiv.org/abs/2109.01652)

The [LLM evaluation arc](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb) develops the missing metric, judge, safety, hallucination, and calibration layers.

## The Decision: What Can Riverside Hand Off Today?

The current evidence supports an architecture decision and a shortlist, not a production checkpoint winner.

### Evidence-aware scorecard

| Candidate | Demonstrated here | Missing before promotion | Status |
| --- | --- | --- | --- |
| Baseline | Control examples and corpus-probe score | Domain/task capability | Reject for Riverside workloads |
| Full-FT continuation | Adapted prose examples | Clean test split, matched strategy comparison, operational cost | Style candidate only |
| Partial-freeze continuation | Adapted prose examples | Same missing evidence as full FT | Style candidate only |
| LoRA continuation | Adapted prose examples and small adapter artifact | Same missing evidence as full FT | Style candidate only |
| SFT LoRA | Instruction-formatted training and qualitative examples | Deterministic task suite, groundedness, safety, latency | Assistant candidate |
| DPO adapter | DPO mechanics on top of SFT | Preference win rate on real editor labels | Do not promote |

### Workload-first handoff

```mermaid
flowchart TD
    Request["Riverside workload"]
    Request --> Facts{"Needs current, citable<br/>manuscript facts?"}
    Facts -->|Yes| Retrieval["Hybrid retrieval + grounded generation<br/>Do not rely on model weights as the database"]
    Facts -->|No| Behavior{"Primary behavior?"}
    Behavior -->|Follow an editor instruction| SFT["SFT-LoRA candidate<br/>Run instruction, safety, and latency gates"]
    Behavior -->|Continue in house style| Style["Continuation candidates<br/>Retrain full/partial/LoRA on one clean split"]
    SFT --> Promote{"All workload gates pass?"}
    Style --> Promote
    Retrieval --> Promote
    Promote -->|Yes| Canary["Versioned canary release"]
    Promote -->|No| Hold["Keep prior release; collect evidence"]
```

This produces three concrete handoffs:

1. **Knowledge base:** build retrieval over the manuscript corpus and require grounded, citable answers. Fine-tuned weights may shape behavior, but they are not the source of truth.
2. **Editing assistant:** carry the SFT-LoRA artifact forward as the leading candidate because its training objective matches instruction following. Do not call it reliable until it passes the evaluation suite.
3. **House-style continuation:** carry full FT, partial freezing, and LoRA forward as candidates. Retrain them under one matched protocol before choosing among them.

The DPO adapter stays out of promotion because this run did not establish a preference improvement.

### Intuition to keep

- **Choose the objective from the workload.** Perplexity on prose cannot select an instruction-following assistant.
- **Choose the parameter strategy from matched quality and operational evidence.** Trainable percentage alone is not a quality, memory, or latency result.
- **Examples generate hypotheses; suites support decisions.** A fluent sample is useful for seeing failure modes, not for writing "Yes" in a scorecard.
- **Retrieval and fine-tuning solve different problems.** Retrieval supplies current evidence; fine-tuning changes persistent behavior.

This closes the fine-tuning arc with an honest shortlist and evaluation plan. [`04-hybrid-search.ipynb`](04-hybrid-search.ipynb) now picks up the knowledge-base branch by finding the right passages across Riverside's catalog.

## From Notebook Shortlist to Production Control Plane

In production, the data-objective choice and parameter-strategy choice become a versioned release decision. A candidate must pass workload-specific **evaluation gates** before promotion: clean test-set perplexity for house-style continuation, instruction pass rate for the editing assistant, preference win rate only when DPO benefit is claimed, and safety/regression checks for every workload.[^1]

"Clean test set" means the file list was fixed before training and excluded from every candidate's training data. The shared corpus probe above does not qualify and is never copied into the gate runner below.

The accepted base model, adapter/checkpoint, tokenizer, dataset fingerprint, code revision, seed, and measured metrics are recorded together in an artifact registry so the release can be reconstructed.[^2] Gates also include p95 latency and cost at the intended batch size and hardware; trainable-parameter percentage does not replace serving benchmarks.[^3]

Promotion should be gradual, with the prior immutable artifact retained as the rollback target. If online quality, latency, cost, or safety thresholds fail, routing returns to that known-good version.[^4]

The cells below define a vendor-neutral manifest and gate runner. They default to `RUN_PRODUCTION_DECISION = False`, so reading or running the notebook does not hash checkpoints, read benchmark files, or write release artifacts.

[^1]: NIST, [AI Risk Management Framework: Measure](https://airc.nist.gov/AI_RMF_Knowledge_Base/Playbook/Measure), recommends documented, repeatable evaluation against deployment-context criteria.
[^2]: MLflow's open-source [Model Registry concepts](https://mlflow.org/docs/latest/ml/model-registry/) illustrate versioned artifacts, lineage, aliases, and controlled promotion.
[^3]: MLCommons, [MLPerf Inference](https://mlcommons.org/benchmarks/inference/), separates serving measurements by scenario because latency and throughput depend on the runtime configuration.
[^4]: Google SRE, [Canarying Releases](https://sre.google/workbook/canarying-releases/), describes comparing a candidate with a known-good release and reverting when the canary degrades.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
from typing import Any, Mapping, Optional


RUN_PRODUCTION_DECISION = False


@dataclass(frozen=True)
class EvaluationPolicy:
    """Workload-specific promotion thresholds; replace defaults with service SLOs."""

    max_perplexity_regression_pct: Optional[float] = None
    min_instruction_pass_rate: Optional[float] = 0.95
    min_preference_win_rate: Optional[float] = None
    min_safety_pass_rate: float = 1.0
    max_p95_latency_ms: float = 1_500.0
    max_cost_per_1k_requests_usd: float = 1.00


@dataclass(frozen=True)
class ProductionDecisionConfig:
    workload: str = "editing-assistant"
    candidate_name: str = "Instruction-tuned (LoRA)"
    candidate_artifact: Path = Path("./checkpoints/instruction-lora")
    rollback_name: str = "previous-production"
    rollback_artifact: Path = Path("./artifacts/production/current")
    benchmark_metrics: Path = Path("./artifacts/production-benchmarks.json")
    registry_dir: Path = Path("./artifacts/finetuning-decisions")
    seed: int = 42
    policy: EvaluationPolicy = EvaluationPolicy()


def set_reproducible_seed(seed: int) -> None:
    """Seed the random sources used by this notebook's PyTorch workflow."""
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_benchmark_metrics(path: Path) -> dict[str, Any]:
    """Load offline quality, safety, latency, and cost measurements."""
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_artifact(path: Path) -> str:
    """Create one deterministic digest for a checkpoint file or directory."""
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for file_path in files:
        relative_path = file_path.name if path.is_file() else file_path.relative_to(path).as_posix()
        digest.update(relative_path.encode("utf-8"))
        with file_path.open("rb") as artifact_file:
            for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def evaluate_release(
    candidate: Mapping[str, float],
    baseline: Mapping[str, float],
    policy: EvaluationPolicy,
) -> dict[str, bool]:
    """Apply only the gates configured for this workload."""
    gates = {
        "instruction": (
            policy.min_instruction_pass_rate is None
            or candidate["instruction_pass_rate"] >= policy.min_instruction_pass_rate
        ),
        "preference": (
            policy.min_preference_win_rate is None
            or candidate["preference_win_rate"] >= policy.min_preference_win_rate
        ),
        "safety": candidate["safety_pass_rate"] >= policy.min_safety_pass_rate,
        "latency": candidate["p95_latency_ms"] <= policy.max_p95_latency_ms,
        "cost": candidate["cost_per_1k_requests_usd"] <= policy.max_cost_per_1k_requests_usd,
    }
    if policy.max_perplexity_regression_pct is not None:
        allowed = baseline["heldout_perplexity"] * (
            1.0 + policy.max_perplexity_regression_pct / 100.0
        )
        gates["perplexity"] = candidate["heldout_perplexity"] <= allowed
    return gates


def build_decision_manifest(
    config: ProductionDecisionConfig,
    benchmark: Mapping[str, Any],
    gates: Mapping[str, bool],
    artifact_digest: str,
) -> dict[str, Any]:
    """Capture the evidence and lineage needed to reproduce or roll back a release."""
    promoted = all(gates.values())
    return {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "workload": config.workload,
        "decision": "promote" if promoted else "rollback",
        "selected_name": config.candidate_name if promoted else config.rollback_name,
        "selected_artifact": str(
            config.candidate_artifact if promoted else config.rollback_artifact
        ),
        "candidate": {
            "name": config.candidate_name,
            "artifact": str(config.candidate_artifact),
            "sha256": artifact_digest,
        },
        "rollback": {
            "name": config.rollback_name,
            "artifact": str(config.rollback_artifact),
        },
        "reproducibility": {
            "seed": config.seed,
            "dataset_fingerprint": benchmark["dataset_fingerprint"],
            "code_revision": benchmark["code_revision"],
            "base_model": MODEL_NAME,
        },
        "policy": asdict(config.policy),
        "metrics": benchmark["candidate"],
        "baseline_metrics": benchmark["baseline"],
        "gates": dict(gates),
    }


def write_decision_manifest(manifest: Mapping[str, Any], registry_dir: Path) -> Path:
    """Write an immutable, content-addressed decision record."""
    registry_dir.mkdir(parents=True, exist_ok=True)
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"))
    decision_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    output_path = registry_dir / f"decision-{decision_id}.json"
    output_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    return output_path

In [ ]:
production_config = ProductionDecisionConfig()

if RUN_PRODUCTION_DECISION:
    set_reproducible_seed(production_config.seed)

    # Release metrics must come from the external benchmark artifact. The shared corpus probe in this
    # notebook is training-contaminated and is intentionally never copied into a production gate.
    benchmark = load_benchmark_metrics(production_config.benchmark_metrics)
    candidate_metrics = dict(benchmark["candidate"])
    benchmark = {**benchmark, "candidate": candidate_metrics}
    gates = evaluate_release(
        candidate_metrics,
        benchmark["baseline"],
        production_config.policy,
    )

    if not production_config.candidate_artifact.exists():
        raise FileNotFoundError(
            f"Candidate artifact not found: {production_config.candidate_artifact}"
        )

    artifact_digest = sha256_artifact(production_config.candidate_artifact)
    manifest = build_decision_manifest(
        production_config,
        benchmark,
        gates,
        artifact_digest,
    )
    manifest_path = write_decision_manifest(manifest, production_config.registry_dir)

    for gate_name, passed in gates.items():
        print(f"{gate_name:>12}: {'PASS' if passed else 'FAIL'}")
    print(f"Decision: {manifest['decision'].upper()} -> {manifest['selected_name']}")
    print(f"Manifest: {manifest_path}")
else:
    print(
        "Production decision workflow is disabled. Set RUN_PRODUCTION_DECISION = True "
        "only after supplying clean benchmark metrics and an explicit rollback artifact."
    )